In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## Importing Required Libraries

In [41]:
import os 
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential,Model
from tensorflow.keras.layers import Dense,Flatten,Conv2D,MaxPooling2D,Dropout,BatchNormalization
from tensorflow.keras.utils import to_categorical,plot_model
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


In [3]:
flowers_data_path = os.path.join("/kaggle/input/flowers-recognition/flowers/")
os.listdir(flowers_data_path)

In [4]:
len(flowers_data_path)

## Loading and preparing Data

In [5]:
data  = []
label = []

size = 128 # Resize to 128 x128

for folder in os.listdir(flowers_data_path):
    for file in os.listdir(os.path.join(flowers_data_path,folder)):
        if file.endswith("jpg"):
            img = cv2.imread(os.path.join(flowers_data_path,folder,file))
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img_rgb,(size,size))
            data.append(img)
            label.append(folder)

In [6]:
len(data),len(label)

In [7]:
data_arr = np.array(data)
label_arr = np.array(label)

## Formattting Data into keras form

In [8]:
encoder = LabelEncoder()
label_arr = encoder.fit_transform(label_arr)

y =  to_categorical(label_arr,5)

x = data_arr/255


In [9]:
x.shape,y.shape

In [10]:
ind =5
plt.imshow(x[ind])
plt.xlabel(label_arr[ind])


In [11]:
 fig = plt.figure(figsize=(10,20))

for i in range(25):
   
    fig.add_subplot(5,5,i+1)
    plt.imshow(x[i])
    plt.xlabel(y[i])
    

## Preprocessing Data

In [12]:
datagen  = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.2
)

## Splitting into train and test

In [13]:
x_train,x_test,y_train,y_test =  train_test_split(x,y,test_size=0.2,random_state=42)

In [14]:
datagen.flow(x_train)

## Model Creation

In [43]:
model = Sequential([
    Conv2D(filters=32,kernel_size=(3,3),activation='relu',input_shape=(128,128,3)),
    MaxPooling2D((2,2)),
    
    
     Conv2D(filters=64,kernel_size=(3,3),activation='relu'),
    MaxPooling2D((2,2)),
    
    
    
     Conv2D(filters=96,kernel_size=(3,3),activation='relu',input_shape=(128,128,3)),
    MaxPooling2D((2,2)),
    
     Conv2D(filters=96,kernel_size=(3,3),activation='relu',input_shape=(128,128,3)),
    MaxPooling2D((2,2))
    
    Flatten(),
    Dense(128,activation='relu'),
    Dense(64,activation='relu'),
    Dense(5,activation='softmax')
])


#### using callbacks to reduce learning_rate

In [44]:
callbacks =[ReduceLROnPlateau(monitor='val_acc',patience=3,factor=0.1)]

In [47]:
model.compile(loss='categorical_crossentropy',optimizer=Adam(learning_rate=0.0001),metrics=['acc'])

In [48]:
model.summary()

In [49]:
plot_model(model)

In [50]:
batch_size=128
epochs=50
history  = model.fit(datagen.flow(x_train,y_train,batch_size=batch_size),
                     epochs=epochs,
                    validation_data=(x_test,y_test),
                     callbacks=callbacks
                    )

## Plotting acc and loss

In [51]:
history_df = pd.DataFrame(history.history)
history_df[['loss','val_loss']].plot(title='loss')
history_df[['acc','val_acc']].plot(title='acc')

## Evaluating Data

In [52]:
loss,acc = model.evaluate(x_test,y_test)
acc

## Making Predictions

In [75]:
def predict(data):
    pred = model.predict(data)
    return np.argmax(pred,axis=1)

In [85]:
predictions = predict(x_test[0].reshape(1,128,128,3))
predictions

In [84]:
x_test[:3].shape

In [100]:
fig= plt.figure(figsize=(10,10))

for i in range(25):
    fig.add_subplot(5,5,i+1)
    plt.imshow(x_test[i])
    plt.xticks([])
    plt.yticks([])
    pred  = os.listdir(flowers_data_path)[predict(x_test[i].reshape(1,128,128,3))[0]]
    true_label = os.listdir(flowers_data_path)[np.argmax(y_test[i])]
    
    if pred ==true_label:
        color = 'green'
    else:
        color ='red'
    
    plt.xlabel(pred,color=color)